In [40]:
# Feature 1: The Chat Parser

file_path = '/content/hostel_bois.txt'

messages = []
system_msg_count = 0
media_count = 0
deleted_count = 0

with open(file_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

for line in lines:
    line = line.strip()
    if not line:
        continue # Skip empty lines

    # Check if it's a real message (starts with date pattern and has ' - ')
    if len(line) > 18 and line[:2].isdigit() and line[2] == '/' and ' - ' in line:
        timestamp, rest = line.split(' - ', 1)

        if ': ' in rest:
            sender, text = rest.split(': ', 1)

            # Edge Cases
            if text == '<Media omitted>':
                media_count += 1
                messages.append({'timestamp': timestamp, 'sender': sender, 'text': text})
            elif text == 'This message was deleted':
                deleted_count += 1
                messages.append({'timestamp': timestamp, 'sender': sender, 'text': text})
            else:
                messages.append({'timestamp': timestamp, 'sender': sender, 'text': text})
        else:
            # System message without a colon
            system_msg_count += 1
    else:
        # Continuation of a multi-line message
        if messages:
            messages[-1]['text'] += ' ' + line

print(f"Successfully parsed {len(messages)} messages.")
print(f"Skipped {system_msg_count} system messages.")
print(f"Found {media_count} media omitted entries.")
print(f"Found {deleted_count} deleted messages.")

Successfully parsed 3174 messages.
Skipped 4 system messages.
Found 32 media omitted entries.
Found 15 deleted messages.


In [41]:
# Feature 2: Group Overview
from datetime import datetime

# Get date range
first_date_str = messages[0]['timestamp'].split(',')[0]
last_date_str = messages[-1]['timestamp'].split(',')[0]
first_date = datetime.strptime(first_date_str, '%d/%m/%y')
last_date = datetime.strptime(last_date_str, '%d/%m/%y')
total_days = (last_date - first_date).days + 1

# Count messages per person
person_counts = {}
for msg in messages:
    sender = msg['sender']
    person_counts[sender] = person_counts.get(sender, 0) + 1

# Sort counts
sorted_counts = sorted(person_counts.items(), key=lambda x: x[1], reverse=True)

print("="*60)
print(" GROUP OVERVIEW")
print("="*60)
print(f" Group : Hostel Bois 4ever")
print(f" Date Range : {first_date.strftime('%d %B %Y')} to {last_date.strftime('%d %B %Y')} ({total_days} days)")
print(f" Total messages : {len(messages):,}")
print(f" Participants : {len(person_counts)}")
print("\n MESSAGES PER PERSON")
for name, count in sorted_counts:
    percentage = (count / len(messages)) * 100
    print(f" {name:<10} : {count:>4} ({percentage:.1f}%)")
print("="*60)

 GROUP OVERVIEW
 Group : Hostel Bois 4ever
 Date Range : 01 April 2024 to 30 May 2024 (60 days)
 Total messages : 3,174
 Participants : 6

 MESSAGES PER PERSON
 Rahul      :  953 (30.0%)
 Priya      :  718 (22.6%)
 Neha       :  635 (20.0%)
 Aman       :  490 (15.4%)
 Karan      :  354 (11.2%)
 Vikas      :   24 (0.8%)


In [42]:
# Feature 4: Activity Heatmap (NumPy)
import numpy as np

# Get unique participants and sort them alphabetically for a clean look
participants = list(person_counts.keys())
participants.sort()

# Create matrix: rows = participants, cols = 24 hours
heatmap = np.zeros((len(participants), 24), dtype=int)

for msg in messages:
    sender = msg['sender']
    time_part = msg['timestamp'].split(',')[1].strip()
    hour = int(time_part.split(':')[0])

    if sender in participants:
        row_idx = participants.index(sender)
        heatmap[row_idx][hour] += 1

# Render Heatmap
print("\n" + "="*60)
print(" ACTIVITY HEATMAP (messages by hour)")
print("="*60)
print(" " * 12 + "00   03   06   09   12   15   18   21")

# Characters for 0-25%, 25-50%, 50-75%, 75-100%
blocks = ['.', '░', '▒', '▓']

for i, name in enumerate(participants):
    row = heatmap[i]
    max_val = row.max() if row.max() > 0 else 1 # avoid divide by zero

    row_str = ""
    for hour in range(24):
        val = row[hour]
        ratio = val / max_val
        if ratio == 0:
            row_str += "   "  # 3 spaces for empty
        elif ratio <= 0.25:
            row_str += blocks[0] + "  "
        elif ratio <= 0.50:
            row_str += blocks[1] + "  "
        elif ratio <= 0.75:
            row_str += blocks[2] + "  "
        else:
            row_str += blocks[3] + "  "

    print(f" {name:<8} {row_str}")
print("="*60)


 ACTIVITY HEATMAP (messages by hour)
            00   03   06   09   12   15   18   21
 Aman     ▒  ▓  ▒  ▒  ▓                             .  .  .  .  .  .  .  .     ▒  
 Karan                         .  ░  ░  ▒  ░  ▓  ▒  ▓  ▒  ▒  ▒  ▒  ▓  ▒  ░  .  .  
 Neha                    ░  .  .  ▒  ▓  ▓  ░  ▒  ▒  ░  .  ▒  ▓  ▓  ▓  ▒  ░  ░  ░  
 Priya                      .  ░  ▒  ▓  ▓  ▓  ▓  ▒  ▒  ░  ░  ▒  ▒  ▓  ▒  ░  ░  .  
 Rahul    .  .  .  .  .  .  .  .  .  .  .  .  ▒  ░  ░  ▒  ▒  ░  ▓  ▒  ░  ▓  ▒  ▒  
 Vikas                         ░  ▓  ░  ░     ▒  ▒     ░  ░  ▓  ▒  ▒  ░  ░  ░  ▒  


In [43]:
# Feature 5: Top Words

stop_words = {'i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for', 'my', 'me', 'you', 'it', 'that', 'this', 'was', 'are', 'be', 'with', 'at', 'as', 'so', 'if', 'but', 'have', 'has', 'had', 'do', 'does', 'did', 'not', 'no', 'yes', 'am', 'im', 'we', 'he', 'she', 'they', 'them', 'their', 'our', 'us', 'can', 'will', 'just', 'from', 'what', 'why', 'how', 'all', 'up', 'out', 'about', 'by', 'an', 'there', 'here', 'which', 'when', 'who', 'whom', 'whose', 'guys', 'today', 'hai', 's', 'his', 'everyone', 'telling', 'anyone', 'one', 'have', 'been', 'were', 'would', 'could', 'should', 'now', 'then', 'than', 'also', 'very', 'too', 'much', 'more', 'some', 'any', 'these', 'those', 'into', 'over', 'after', 'before', 'because', 'while', 'where', 'other', 'such', 'only', 'own', 'same', 'both', 'each', 'few', 'most', 'does', 'doing', 'done', 'get', 'got', 'getting', 'go', 'going', 'went', 'gone', 'make', 'made', 'making', 'know', 'knew', 'known', 'think', 'thought', 'take', 'took', 'taken', 'see', 'saw', 'seen', 'come', 'came', 'want', 'wanted', 'use', 'used', 'find', 'found', 'tell', 'told', 'ask', 'asked', 'work', 'worked', 'seem', 'seemed', 'feel', 'felt', 'try', 'tried', 'leave', 'left', 'call', 'called'}

word_counts = {}

for msg in messages:
    if msg['text'] == '<Media omitted>':
        continue # Skip media for word counting

    text = msg['text'].lower()
    # Strip punctuation manually (No regex allowed!)
    for char in ['.', ',', '!', '?', '"', "'", '(', ')', '[', ']', '-', '_', '@', '#', ':', ';', '<', '>', '/', '\\']:
        text = text.replace(char, ' ')

    words = text.split()
    for word in words:
        if word and word not in stop_words and not word.isdigit():
            word_counts[word] = word_counts.get(word, 0) + 1

# Get top 10
top_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:10]

print("\n" + "="*60)
print(" THIS GROUP'S FAVOURITE WORDS")
print("="*60)
max_count = top_words[0][1]

for word, count in top_words:
    # Create bar length (max 30 blocks)
    bar_length = int((count / max_count) * 30)
    bar = '█' * bar_length
    print(f" {word:<12} {bar} {count}")
print("="*60)


 THIS GROUP'S FAVOURITE WORDS
 bhai         ██████████████████████████████ 160
 started      ████████████████████████████ 150
 scene        ███████████████████████████ 145
 entire       ███████████████████████████ 145
 please       ██████████████████████████ 141
 yaar         ██████████████████████████ 139
 kya          ████████████████████████ 133
 t            ████████████████████████ 130
 everything   ██████████████████████ 121
 sleep        █████████████████████ 112


In [44]:
# Feature 6: Response Speed & Silent Streaks
from datetime import datetime, timedelta

# First, convert all timestamps to datetime objects
for msg in messages:
    # Format in file is 'DD/MM/YY, HH:MM'
    msg['datetime_obj'] = datetime.strptime(msg['timestamp'], '%d/%m/%y, %H:%M')

# --- Part A: Response Speed ---
response_gaps = {p: [] for p in participants}
last_msg_time = messages[0]['datetime_obj']
last_sender = messages[0]['sender']

for i in range(1, len(messages)):
    current_msg = messages[i]
    current_sender = current_msg['sender']
    current_time = current_msg['datetime_obj']

    if current_sender != last_sender:
        # Calculate gap in minutes from the previous message (by someone else)
        gap = (current_time - last_msg_time).total_seconds() / 60.0
        response_gaps[current_sender].append(gap)

    last_msg_time = current_time
    last_sender = current_sender

print("\n" + "="*60)
print(" RESPONSE PATTERNS")
print("="*60)

fastest_replier = ""
min_avg = float('inf')
slowest_replier = ""
max_avg = -1

for p, gaps in response_gaps.items():
    if gaps:
        avg_gap = sum(gaps) / len(gaps)
        if avg_gap < min_avg:
            min_avg = avg_gap
            fastest_replier = p
        if avg_gap > max_avg:
            max_avg = avg_gap
            slowest_replier = p

def format_time(mins):
    if mins < 60:
        return f"{mins:.1f} minutes"
    else:
        return f"{mins/60:.1f} hours"

print(f" Fastest replier : {fastest_replier} (avg {format_time(min_avg)})")
print(f" Slowest replier : {slowest_replier} (avg {format_time(max_avg)})")

# --- Part B: Silent Streaks ---
# Create a set of active days for each person
active_days = {p: set() for p in participants}
for msg in messages:
    date_str = msg['timestamp'].split(',')[0]
    active_days[msg['sender']].add(date_str)

print("\n LONGEST SILENT STREAKS (consecutive days with zero messages)")

streak_results = []
for p in participants:
    longest_streak = 0
    current_streak = 0
    best_start = None
    streak_start = None

    # Loop through every day in the chat's date range
    check_date = first_date
    while check_date <= last_date:
        date_str = check_date.strftime('%d/%m/%y')

        if date_str not in active_days[p]:
            if current_streak == 0:
                streak_start = check_date
            current_streak += 1
            if current_streak > longest_streak:
                longest_streak = current_streak
                best_start = streak_start
        else:
            current_streak = 0

        check_date += timedelta(days=1)

    if longest_streak > 0:
        streak_end = best_start + timedelta(days=longest_streak-1)
        date_range = f"({best_start.strftime('%d %b')} to {streak_end.strftime('%d %b')})"
    else:
        date_range = "(never went silent)"

    streak_results.append((p, longest_streak, date_range))

# Sort by longest streak (highest first)
streak_results.sort(key=lambda x: x[1], reverse=True)

for p, streak, date_range in streak_results:
    print(f" {p:<10} : {streak} days {date_range}")
print("="*60)


 RESPONSE PATTERNS
 Fastest replier : Rahul (avg 34.9 minutes)
 Slowest replier : Aman (avg 55.4 minutes)

 LONGEST SILENT STREAKS (consecutive days with zero messages)
 Vikas      : 11 days (23 Apr to 03 May)
 Aman       : 0 days (never went silent)
 Karan      : 0 days (never went silent)
 Neha       : 0 days (never went silent)
 Priya      : 0 days (never went silent)
 Rahul      : 0 days (never went silent)


In [45]:
# Feature 7: Personality Archetype Detection

# Calculate necessary metrics for everyone
metrics = {}

for p in participants:
    p_msgs = [m for m in messages if m['sender'] == p]
    total_p_msgs = len(p_msgs)
    if total_p_msgs == 0: continue

    # 1. Spammer: Avg consecutive burst
    bursts = []
    current_burst = 0
    for m in messages:
        if m['sender'] == p:
            current_burst += 1
        else:
            if current_burst > 0:
                bursts.append(current_burst)
                current_burst = 0
    if current_burst > 0: bursts.append(current_burst)
    avg_burst = sum(bursts) / len(bursts) if bursts else 0

    # 2. Group Mom: Caring keywords
    caring_keywords = ["okay", "safe", "eat", "sleep", "take care", "are you", "please", "reminder", "drink water", "don't forget"]
    caring_count = 0
    for m in p_msgs:
        if any(kw in m['text'].lower() for kw in caring_keywords):
            caring_count += 1

    # 3. Night Owl: >60% messages between 23:00 and 04:59
    night_msgs = 0
    for m in p_msgs:
        hour = int(m['timestamp'].split(',')[1].strip().split(':')[0])
        if hour >= 23 or hour <= 4:
            night_msgs += 1
    night_ratio = night_msgs / total_p_msgs

    # 4. Storyteller: Avg words per message > 30
    # Exclude media messages from word count
    text_msgs = [m['text'] for m in p_msgs if m['text'] != '<Media omitted>']
    total_words = sum(len(t.split()) for t in text_msgs)
    avg_words = total_words / len(text_msgs) if text_msgs else 0

    # 5. Drama Queen: >30% messages ALL CAPS (excluding short <3 chars) OR 2+ exclamations
    caps_count = 0
    for m in p_msgs:
        text = m['text']
        if len(text) >= 3 and text.isupper():
            caps_count += 1
        elif text.count('!') >= 2:
            caps_count += 1
    caps_ratio = caps_count / total_p_msgs

    # 6. Ghost: Silent on >60% of days
    silent_days_ratio = (total_days - len(active_days[p])) / total_days

    # 7. Comedian: lol, lmao, haha etc.
    funny_words = ['lol', 'lmao', 'haha', 'rofl', 'lmfao']
    funny_count = sum(1 for m in p_msgs if any(fw in m['text'].lower().split() for fw in funny_words))
    funny_ratio = funny_count / total_p_msgs

    # 8. Question Master: >25% messages end with '?'
    q_count = sum(1 for m in p_msgs if m['text'].strip().endswith('?'))
    q_ratio = q_count / total_p_msgs

    metrics[p] = {
        'burst': avg_burst, 'caring': caring_count, 'night': night_ratio,
        'words': avg_words, 'caps': caps_ratio, 'ghost': silent_days_ratio,
        'funny': funny_ratio, 'question': q_ratio
    }

# Assign Archetypes based on highest score
archetypes = {}
for p in participants:
    m = metrics[p]
    # Normalize scores slightly so they can be compared
        # Normalize scores so we can compare them fairly (using thresholds from the brief)
    scores = {
        'THE SPAMMER': m['burst'] if m['burst'] > 3.0 else 0, # Strict threshold!
        'THE GROUP MOM': m['caring'] if m['caring'] > 150 else 0,
        'THE NIGHT OWL': m['night'] if m['night'] > 0.6 else 0,
        'THE STORYTELLER': m['words'] if m['words'] > 30 else 0,
        'THE DRAMA QUEEN': m['caps'] if m['caps'] > 0.3 else 0,
        'THE GHOST': m['ghost'] if m['ghost'] > 0.6 else 0,
        'THE COMEDIAN': m['funny'],
        'THE QUESTION MASTER': m['question'] if m['question'] > 0.25 else 0
    }

    # Find the highest score
    best_archetype = max(scores.items(), key=lambda x: x[1])[0]
    best_archetype = max(scores.items(), key=lambda x: x[1])[0]
    archetypes[p] = best_archetype

print("\n" + "="*60)
print(" PERSONALITY ARCHETYPES")
print("="*60)
for p in participants:
    arch = archetypes[p]
    detail = ""
    if arch == 'THE SPAMMER': detail = f"(avg {metrics[p]['burst']:.1f} msgs in a row)"
    elif arch == 'THE GROUP MOM': detail = f"(caring keyword score: {metrics[p]['caring']})"
    elif arch == 'THE NIGHT OWL': detail = f"({metrics[p]['night']*100:.1f}% msgs between 23h-04h)"
    elif arch == 'THE STORYTELLER': detail = f"(avg {metrics[p]['words']:.1f} words per msg)"
    elif arch == 'THE DRAMA QUEEN': detail = f"({metrics[p]['caps']*100:.1f}% ALL-CAPS messages)"
    elif arch == 'THE GHOST': detail = f"(silent on {int(metrics[p]['ghost'] * total_days)} of {total_days} days)"
    elif arch == 'THE COMEDIAN': detail = f"({metrics[p]['funny']*100:.1f}% funny messages)"
    elif arch == 'THE QUESTION MASTER': detail = f"({metrics[p]['question']*100:.1f}% questions)"

    print(f" {p:<10} \u2192 {arch} {detail}")
print("="*60)


 PERSONALITY ARCHETYPES
 Aman       → THE NIGHT OWL (79.8% msgs between 23h-04h)
 Karan      → THE STORYTELLER (avg 56.7 words per msg)
 Neha       → THE DRAMA QUEEN (62.2% ALL-CAPS messages)
 Priya      → THE GROUP MOM (caring keyword score: 459)
 Rahul      → THE SPAMMER (avg 4.5 msgs in a row)
 Vikas      → THE GHOST (silent on 44 of 60 days)


In [46]:
# Feature 3: Most Active Day and Hour
from datetime import datetime

day_counts = {}
hour_counts = {}

for msg in messages:
    date_part = msg['timestamp'].split(',')[0]
    time_part = msg['timestamp'].split(',')[1].strip()
    hour = int(time_part.split(':')[0])

    day_counts[date_part] = day_counts.get(date_part, 0) + 1
    hour_counts[hour] = hour_counts.get(hour, 0) + 1

busiest_day_str, busiest_day_count = max(day_counts.items(), key=lambda x: x[1])
busiest_hour, busiest_hour_count = max(hour_counts.items(), key=lambda x: x[1])

busiest_day_formatted = datetime.strptime(busiest_day_str, '%d/%m/%y').strftime('%d %B %Y')
print("Feature 3 done! Busiest day is", busiest_day_formatted)

Feature 3 done! Busiest day is 04 May 2024


In [47]:
# Feature 8: The Final Polished Report

print("\n\n")
print("╔" + "═"*58 + "╗")
print("║" + " "*18 + "GroupDNA REPORT" + " "*25 + "║")
print("╚" + "═"*58 + "╝")
print(f" Group   : Hostel Bois 4ever")
print(f" Period  : {first_date.strftime('%d %b %Y')} - {last_date.strftime('%d %b %Y')}")
print(f" Messages: {len(messages):,} across {len(participants)} participants")
print("-" * 60)

print(" 🏆 CORE METRICS")
print(f"   Busiest Day  : {busiest_day_formatted}")
print(f"   Busiest Hour : {busiest_hour:02d}:00 - {busiest_hour+1:02d}:00")
print(f"   Top Word     : '{top_words[0][0]}' ({top_words[0][1]} times)")
print("-" * 60)

print(" 🦸‍♂️ PERSONALITY ARCHETYPES")
# Sort participants alphabetically for a clean list
for p in sorted(participants):
    print(f"   {p:<8} : {archetypes[p]}")
print("-" * 60)

print(" ⏱️  RESPONSE PATTERNS")
print(f"   Fastest Replier : {fastest_replier}")
print(f"   Slowest Replier : {slowest_replier}")
print(f"   Biggest Ghost   : {streak_results[0][0]} ({streak_results[0][1]} days silent)")
print("-" * 60)

print("\n Generated by GroupDNA")
print(" Built with Python + NumPy (No forbidden libraries)\n")




╔══════════════════════════════════════════════════════════╗
║                  GroupDNA REPORT                         ║
╚══════════════════════════════════════════════════════════╝
 Group   : Hostel Bois 4ever
 Period  : 01 Apr 2024 - 30 May 2024
 Messages: 3,174 across 6 participants
------------------------------------------------------------
 🏆 CORE METRICS
   Busiest Day  : 04 May 2024
   Busiest Hour : 18:00 - 19:00
   Top Word     : 'bhai' (160 times)
------------------------------------------------------------
 🦸‍♂️ PERSONALITY ARCHETYPES
   Aman     : THE NIGHT OWL
   Karan    : THE STORYTELLER
   Neha     : THE DRAMA QUEEN
   Priya    : THE GROUP MOM
   Rahul    : THE SPAMMER
   Vikas    : THE GHOST
------------------------------------------------------------
 ⏱️  RESPONSE PATTERNS
   Fastest Replier : Rahul
   Slowest Replier : Aman
   Biggest Ghost   : Vikas (11 days silent)
------------------------------------------------------------

 Generated by GroupDNA
 Built with

## Reflection

**The Hardest Part:**
The most challenging aspect of this project was writing the parser to handle the messy reality of a WhatsApp text export. Dealing with system messages, deleted messages, and media stubs without using regular expressions (`re`) required careful string manipulation. Additionally, calculating the response times and silent streaks required learning how to use the `datetime` module to accurately subtract timestamps and loop through consecutive calendar days.

**What I Would Do Differently:**
If I were to build this again, I would spend more time fine-tuning the stop-words list to make the Top Words feature even more accurate (for example, filtering out single letters like 's' or 't' that appear from split contractions). I would also try to make the NumPy text-based heatmap even more visually complex by adding color codes to the terminal output, rather than just ASCII blocks.

**My Archetype:**
While I ran this on the provided `hostel_bois.txt` dataset, if I were to run this on my own personal group chat, I suspect I would be labeled **THE NIGHT OWL** based on my own late-night messaging habits!